# Agent offline evaluation: custom evaluators

Two domain-specific custom evaluators for the Contoso Private Banking knowledge base:

- **AnswerLengthEvaluator** - checks responses fall within a target length range
- **CitationEvaluator** - checks responses reference plausible knowledge-base citations (research notes, market commentary, regulatory memos, IPS documents, or synthetic ISINs). Aria's system prompt explicitly instructs her to reproduce citations of the form `research/<id>` / `market_commentary/<id>` / `ips/<client_id>` - this evaluator audits that contract.

In [1]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown
from azure.ai.evaluation import evaluate
from dotenv import load_dotenv

## Environment

In [2]:
repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

lab_dir        = repo_root / '08-agents' / '08-06-agent-offline-evaluation'
test_data_path = lab_dir / 'test_data.jsonl'

## AnswerLengthEvaluator

Checks whether the response character length is within an acceptable range.

In [3]:
class AnswerLengthEvaluator:
    def __init__(self, min_length=50, max_length=1500):
        self.min_length = min_length
        self.max_length = max_length

    def __call__(self, response, **kwargs):
        length = len(response)
        return {
            'answer_length':      length,
            'answer_length_pass': self.min_length <= length <= self.max_length,
        }

## CitationEvaluator

Checks whether the response cites any plausible knowledge-base reference from the Contoso Private Banking corpus - research notes (`research/res-XXX`), market commentary (`market_commentary/mc-XXX`), regulatory memos (`regulatory/reg-XXX`), IPS documents (`ips/cli-XXX`), CRM events (`crm/crm-XXX`), or synthetic ISINs (`XX0000000XXX`). Aria's system prompt instructs her to reproduce citations of these forms, so this evaluator audits that contract.

In [4]:
import re

# Citation patterns the Contoso Private Banking corpus uses. These match what
# aria's system prompt instructs her to reproduce in her replies.
_CITATION_PATTERNS = [
    re.compile(r'research/res-\d{3}'),         # research notes (factsheets, thematic notes)
    re.compile(r'market_commentary/mc-\d{3}'),  # market commentary
    re.compile(r'regulatory/reg-\d{3}'),       # regulatory memos
    re.compile(r'ips/cli-\d{3}'),               # IPS documents
    re.compile(r'crm/crm-\d{3}'),               # CRM events
    re.compile(r'\bXX0{8}\d{4}\b'),            # synthetic ISINs (documentation range)
    re.compile(r'\b(?:res|mc|reg|crm)-\d{3}\b'),  # bare IDs (less strict - agent may shorten)
]


class CitationEvaluator:
    """Checks the response references at least one plausible CPB knowledge-base citation."""

    def __init__(self, patterns=None):
        self.patterns = patterns or _CITATION_PATTERNS

    def __call__(self, response, **kwargs):
        cited: set[str] = set()
        for p in self.patterns:
            cited.update(p.findall(response))
        return {
            'citation_count':  len(cited),
            'citation_pass':   len(cited) > 0,
            'citations_found': sorted(cited),
        }


## Load test data

In [5]:
test_records = []
with open(test_data_path) as f:
    for line in f:
        line = line.strip()
        if line:
            test_records.append(json.loads(line))

if not test_records:
    raise RuntimeError('test_data.jsonl is empty. Run 08-05-01 first.')

print(f'Loaded {len(test_records)} records')

Loaded 5 records


## Run custom evaluators

In [6]:
custom_results = evaluate(
    data=str(test_data_path),
    evaluators={
        'answer_length': AnswerLengthEvaluator(min_length=50, max_length=1500),
        'citation':      CitationEvaluator(),
    },
    evaluator_config={
        'answer_length': {'column_mapping': {'response': '${data.response}'}},
        'citation':      {'column_mapping': {'response': '${data.response}'}},
    },
)

print('Custom metrics:', custom_results.get('metrics', {}))

2026-05-11 12:49:52 +0200 127945263326912 execution.bulk     INFO     Finished 2 / 5 lines.
2026-05-11 12:49:52 +0200 127945263326912 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-05-11 12:49:52 +0200 127945271719616 execution.bulk     INFO     Finished 3 / 5 lines.
2026-05-11 12:49:52 +0200 127945263326912 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:49:52 +0200 127945271719616 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-05-11 12:49:52 +0200 127945263326912 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-05-11 12:49:52 +0200 127945271719616 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:49:52 +0200 127945271719616 execution.bulk     INFO     Average execution time for comp

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "answer_length_20260511_104952_255254"
Run status: "Completed"
Start time: "2026-05-11 10:49:52.255254+00:00"
Duration: "0:00:01.002309"

======= Run Summary =======

Run name: "citation_20260511_104952_255775"
Run status: "Completed"
Start time: "2026-05-11 10:49:52.255775+00:00"
Duration: "0:00:01.002022"

======= Combined Run Summary (Per Evaluator) =======

{
    "answer_length": {
        "status": "Completed",
        "duration": "0:00:01.002309",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "citation": {
        "status": "Completed",
        "duration": "0:00:01.002022",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}


Custom metrics: {'answer_length.answer_length': 1100.2,

## Display tabular results

In [7]:
rows = custom_results.get('rows', [])
if rows:
    display(Markdown('### Custom Evaluator Row Results'))
    display_data = []
    for i, row in enumerate(rows):
        query = row.get('inputs.query', '')
        display_data.append({
            '#':           i + 1,
            'Query':       query[:50] + '...' if len(query) > 50 else query,
            'Length':      row.get('outputs.answer_length.answer_length', 'N/A'),
            'Length Pass': row.get('outputs.answer_length.answer_length_pass', 'N/A'),
            'Citations':   row.get('outputs.citation.citation_count', 'N/A'),
            'Cite Pass':   row.get('outputs.citation.citation_pass', 'N/A'),
        })
    df = pd.DataFrame(display_data)
    display(df.style.hide(axis='index'))

### Custom Evaluator Row Results

#,Query,Length,Length Pass,Citations,Cite Pass
1,I have a 9am with the Berger family for their quar...,1937,False,12,True
2,Show me the Lindemann family office portfolio drif...,851,True,1,True
3,Anything been written recently about AI infrastruc...,1252,True,6,True
4,Summarise what is been happening on the Riedi pens...,892,True,0,False
5,Get me the FINMA sustainability disclosure status ...,569,True,3,True
